In [ ]:
"""
Huff-based preferred LPG distributor assignment per inhabited 1 km pixel (Nigeria)
using SHORTEST-PATH travel time on a friction graph.

What this script does
---------------------
This script computes, for each inhabited pixel, the preferred LPG distributor for:
1) walking population
2) motorized population

The winner is selected with a Huff-style gravity score:

    score_ij = A_j / (T_ij + eps)^beta

where:
- A_j = distributor attractiveness
- T_ij = shortest-path travel time (minutes) from pixel i to distributor j
- beta = distance-decay exponent (default 2.0)
- eps = small stabilization constant

Important implementation choice
------------------------------
Travel time is computed from a raster graph using shortest-path (Dijkstra) on a
4-neighbor network weighted by friction (minutes per meter -> minutes per cell edge).
No straight-line approximation is used.

Scalability strategy
--------------------
To keep runtime manageable on large grids:
- process only inhabited pixels (from Population.tif)
- build candidate reseller sets progressively (10x10, 15x15, 20x20, ...) until at least
  MIN_CANDIDATES = 4
- run shortest-path from one pixel source with a finite time limit derived from candidates
- for motorized mode, extend candidate set with extra nearest resellers
- print progress and ETA only when estimated total runtime exceeds ~5 minutes

Outputs
-------
Main output: GPKG point layer with one point per inhabited pixel center and fields:
- car_share, walk_share
- best_fid_walk, best_time_walk_min
- best_fid_car, best_time_car_min
- best_place_id_walk, best_place_id_car

Optional outputs: support rasters for shares, winner ids, and winner times.
"""

from __future__ import annotations

import time
from typing import Iterable
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra

import rasterio
from rasterio.warp import reproject, Resampling

# =====================================================================
# USER PARAMETERS
# =====================================================================
DATA_DIR = "dataset_big"

# Inputs
RESELLER_GPKG = f"{DATA_DIR}/full_lpg_chain_nig_3857.gpkg"
RESELLER_LAYER = "resell_and_filling"
POPULATION_RASTER = f"{DATA_DIR}/Population.tif"
CAR_SHARE_RASTER = f"{DATA_DIR}/vehicles_allocation_share.tif"
WALK_FRICTION_RASTER = f"{DATA_DIR}/friction_walk.tif"
MOTO_FRICTION_RASTER = f"{DATA_DIR}/friction_moto.tif"

# Outputs
OUTPUT_GPKG = f"{DATA_DIR}/huff_preferred_distributor_per_pixel.gpkg"
OUTPUT_GPKG_LAYER = "pixel_preference"

OUTPUT_RASTER_CAR_SHARE = f"{DATA_DIR}/huff_car_share.tif"
OUTPUT_RASTER_WALK_SHARE = f"{DATA_DIR}/huff_walk_share.tif"
OUTPUT_RASTER_BEST_FID_WALK = f"{DATA_DIR}/huff_best_fid_walk.tif"
OUTPUT_RASTER_BEST_TIME_WALK = f"{DATA_DIR}/huff_best_time_walk_min.tif"
OUTPUT_RASTER_BEST_FID_CAR = f"{DATA_DIR}/huff_best_fid_car.tif"
OUTPUT_RASTER_BEST_TIME_CAR = f"{DATA_DIR}/huff_best_time_car_min.tif"
OUTPUT_LOOKUP_CSV = f"{DATA_DIR}/huff_reseller_lookup.csv"

# Required reseller columns
FID_COLUMN = "fid"
PLACE_ID_COLUMN = "place_id"
ATTRACTIVENESS_COLUMN = "attractiveness"

# Huff parameters
BETA = 2.0
EPS = 1e-6
MIN_ATTRACTIVENESS = 1e-6

# Population rules
MIN_POP_PER_PIXEL = 0.0

# Car-share interpretation
CAR_SHARE_IS_PERCENT = True
CAR_SHARE_ZERO_THRESHOLD = 1e-9

# Candidate strategy
MIN_CANDIDATES = 4
WINDOW_SIZES = [10, 15, 20, 25, 30, 40, 50]
EXTRA_CANDIDATES_CAR = 6

# Graph assumptions
CELL_SIZE_METERS = 1000.0  # 1km grid
USE_8_NEIGHBORS = False

# Progress
PROGRESS_EVERY = 2000
MAX_PIXELS_DEBUG = None  # set int for quick testing

# Outputs flags
SAVE_SUPPORT_RASTERS = True

# Nodata conventions
NODATA_FLOAT = -9999.0
NODATA_INT = -1

# =====================================================================
# HELPERS
# =====================================================================
def _read_raster(path: str):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32, copy=False)
        profile = src.profile.copy()
        nodata = src.nodata
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr).astype(np.float32)
    return arr, profile


def _align_to_reference(path: str, ref_profile: dict, resampling: Resampling) -> np.ndarray:
    with rasterio.open(path) as src:
        dst = np.full((ref_profile["height"], ref_profile["width"]), np.nan, dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=ref_profile["transform"],
            dst_crs=ref_profile["crs"],
            dst_nodata=np.nan,
            resampling=resampling,
        )
    return dst


def _safe_attractiveness(values: Iterable) -> np.ndarray:
    arr = pd.to_numeric(pd.Series(values), errors="coerce").astype(np.float64).to_numpy()
    arr = np.where(np.isfinite(arr), arr, MIN_ATTRACTIVENESS)
    arr = np.maximum(arr, MIN_ATTRACTIVENESS)
    return arr


def _to_fraction(arr: np.ndarray) -> np.ndarray:
    out = arr.astype(np.float32, copy=True)
    if CAR_SHARE_IS_PERCENT:
        out = out / 100.0
    out = np.where(np.isfinite(out), out, 0.0)
    out = np.clip(out, 0.0, 1.0)
    return out


def _eta(seconds: float) -> str:
    if (not np.isfinite(seconds)) or seconds < 0:
        return "n/a"
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    if h > 0:
        return f"{h}h {m}m {s}s"
    return f"{m}m {s}s"


def _score_huff(a: float, tmin: float) -> float:
    if (not np.isfinite(tmin)) or (tmin <= 0):
        return -np.inf
    return float(a / ((tmin + EPS) ** BETA))


def _write_float_raster(path: str, array: np.ndarray, ref_profile: dict):
    profile = ref_profile.copy()
    profile.update(dtype="float32", count=1, nodata=NODATA_FLOAT, compress="lzw")
    out = np.where(np.isfinite(array), array, NODATA_FLOAT).astype(np.float32)
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(out, 1)


def _write_int_raster(path: str, array: np.ndarray, ref_profile: dict):
    profile = ref_profile.copy()
    profile.update(dtype="int32", count=1, nodata=NODATA_INT, compress="lzw")
    out = np.where(np.isfinite(array), array, NODATA_INT).astype(np.int32)
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(out, 1)


# =====================================================================
# LOAD / ALIGN INPUTS
# =====================================================================
t0 = time.time()
print("[1/8] Loading population reference raster...")
pop, ref_profile = _read_raster(POPULATION_RASTER)
transform = ref_profile["transform"]
crs = ref_profile["crs"]
height, width = pop.shape
print(f"Grid: {width} x {height}, CRS={crs}")

print("[2/8] Aligning car share and frictions to reference grid...")
car_share_raw = _align_to_reference(CAR_SHARE_RASTER, ref_profile, Resampling.nearest)
walk_friction = _align_to_reference(WALK_FRICTION_RASTER, ref_profile, Resampling.bilinear)
moto_friction = _align_to_reference(MOTO_FRICTION_RASTER, ref_profile, Resampling.bilinear)

car_share = _to_fraction(car_share_raw)
walk_share = (1.0 - car_share).astype(np.float32)

walk_friction = np.where(walk_friction > 0, walk_friction, np.nan).astype(np.float32)
moto_friction = np.where(moto_friction > 0, moto_friction, np.nan).astype(np.float32)

print(f"Walk friction range (min/m): {np.nanmin(walk_friction):.6f} .. {np.nanmax(walk_friction):.6f}")
print(f"Moto friction range (min/m): {np.nanmin(moto_friction):.6f} .. {np.nanmax(moto_friction):.6f}")

# =====================================================================
# LOAD RESELLERS
# =====================================================================
print("[3/8] Loading reseller points...")
resellers = gpd.read_file(RESELLER_GPKG, layer=RESELLER_LAYER)
if resellers.empty:
    raise RuntimeError("Reseller layer is empty.")
if resellers.crs != crs:
    resellers = resellers.to_crs(crs)

for col in [FID_COLUMN, ATTRACTIVENESS_COLUMN]:
    if col not in resellers.columns:
        raise KeyError(f"Missing column '{col}' in reseller layer.")

resellers = resellers[resellers.geometry.notna()].copy()
resellers = resellers[resellers.geometry.geom_type.isin(["Point"])].copy()
if resellers.empty:
    raise RuntimeError("No point geometries in reseller layer.")

resellers[ATTRACTIVENESS_COLUMN] = _safe_attractiveness(resellers[ATTRACTIVENESS_COLUMN])

r_rows, r_cols = rasterio.transform.rowcol(
    transform, resellers.geometry.x.values, resellers.geometry.y.values
 )
r_rows = np.asarray(r_rows, dtype=np.int32)
r_cols = np.asarray(r_cols, dtype=np.int32)

inside = (r_rows >= 0) & (r_rows < height) & (r_cols >= 0) & (r_cols < width)
resellers = resellers.loc[inside].copy()
r_rows = r_rows[inside]
r_cols = r_cols[inside]

if resellers.empty:
    raise RuntimeError("No reseller is inside raster extent.")

reseller_fid = pd.to_numeric(resellers[FID_COLUMN], errors="coerce").fillna(-1).astype(np.int32).to_numpy()
reseller_attr = resellers[ATTRACTIVENESS_COLUMN].astype(np.float64).to_numpy()
if PLACE_ID_COLUMN in resellers.columns:
    reseller_place = resellers[PLACE_ID_COLUMN].astype(str).to_numpy()
else:
    reseller_place = np.array(["" for _ in range(len(resellers))], dtype=object)

coords_rc = np.column_stack([r_rows, r_cols]).astype(np.float64)
reseller_tree = cKDTree(coords_rc)
print(f"Resellers on grid: {len(resellers)}")

# =====================================================================
# BUILD SHORTEST-PATH GRAPHS (walk and motorized)
# =====================================================================
print("[4/8] Building graph topology from valid cells...")
valid = np.isfinite(pop) & np.isfinite(walk_friction) & np.isfinite(moto_friction)
node_id = -np.ones((height, width), dtype=np.int32)
vr, vc = np.where(valid)
n_nodes = len(vr)
if n_nodes == 0:
    raise RuntimeError("No valid nodes available for graph.")
node_id[vr, vc] = np.arange(n_nodes, dtype=np.int32)
print(f"Valid graph nodes: {n_nodes:,}")

if USE_8_NEIGHBORS:
    neighbors = [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]
else:
    neighbors = [(-1,0),(1,0),(0,-1),(0,1)]

edge_i = []
edge_j = []
edge_cost_walk = []
edge_cost_moto = []

diag_factor = np.sqrt(2.0)

for r, c in zip(vr, vc):
    n0 = node_id[r, c]
    fw0 = walk_friction[r, c]
    fm0 = moto_friction[r, c]
    for dr, dc in neighbors:
        rr = r + dr
        cc = c + dc
        if rr < 0 or rr >= height or cc < 0 or cc >= width:
            continue
        n1 = node_id[rr, cc]
        if n1 < 0:
            continue
        fw1 = walk_friction[rr, cc]
        fm1 = moto_friction[rr, cc]
        if (not np.isfinite(fw1)) or (not np.isfinite(fm1)):
            continue

        step_m = CELL_SIZE_METERS
        if dr != 0 and dc != 0:
            step_m *= diag_factor

        cw = 0.5 * (fw0 + fw1) * step_m
        cm = 0.5 * (fm0 + fm1) * step_m
        if cw <= 0 or cm <= 0:
            continue

        edge_i.append(n0)
        edge_j.append(n1)
        edge_cost_walk.append(float(cw))
        edge_cost_moto.append(float(cm))

graph_walk = csr_matrix((edge_cost_walk, (edge_i, edge_j)), shape=(n_nodes, n_nodes))
graph_moto = csr_matrix((edge_cost_moto, (edge_i, edge_j)), shape=(n_nodes, n_nodes))
print(f"Directed edges: {len(edge_i):,}")

# map resellers to graph nodes
reseller_node = node_id[r_rows, r_cols]
valid_res = reseller_node >= 0
if not np.any(valid_res):
    raise RuntimeError("No reseller mapped to valid graph node.")
reseller_node = reseller_node[valid_res]
reseller_fid = reseller_fid[valid_res]
reseller_attr = reseller_attr[valid_res]
reseller_place = reseller_place[valid_res]
r_rows = r_rows[valid_res]
r_cols = r_cols[valid_res]
coords_rc = np.column_stack([r_rows, r_cols]).astype(np.float64)
reseller_tree = cKDTree(coords_rc)

# =====================================================================
# INHABITED PIXELS
# =====================================================================
print("[5/8] Preparing inhabited pixel list...")
inhabited = np.isfinite(pop) & (pop > MIN_POP_PER_PIXEL) & valid
pix_rows, pix_cols = np.where(inhabited)
n_pix = len(pix_rows)
if n_pix == 0:
    raise RuntimeError("No inhabited pixels found.")
if MAX_PIXELS_DEBUG is not None:
    n_use = min(MAX_PIXELS_DEBUG, n_pix)
    pix_rows = pix_rows[:n_use]
    pix_cols = pix_cols[:n_use]
    n_pix = n_use
    print(f"DEBUG mode active: {n_pix} pixels")
print(f"Inhabited pixels to process: {n_pix:,}")

best_fid_walk = np.full((height, width), NODATA_INT, dtype=np.int32)
best_time_walk = np.full((height, width), np.nan, dtype=np.float32)
best_fid_car = np.full((height, width), NODATA_INT, dtype=np.int32)
best_time_car = np.full((height, width), np.nan, dtype=np.float32)

fid_to_place = {int(f): str(p) for f, p in zip(reseller_fid, reseller_place)}

def _candidate_idx_progressive(r: int, c: int) -> np.ndarray:
    idx = np.array([], dtype=np.int32)
    for w in WINDOW_SIZES:
        half = max(1, w // 2)
        found = reseller_tree.query_ball_point([r, c], r=half, p=np.inf)
        if len(found) >= MIN_CANDIDATES:
            idx = np.asarray(found, dtype=np.int32)
            break
        if len(found) > len(idx):
            idx = np.asarray(found, dtype=np.int32)

    if idx.size < MIN_CANDIDATES:
        k = min(max(MIN_CANDIDATES, EXTRA_CANDIDATES_CAR), len(coords_rc))
        _, nn = reseller_tree.query([r, c], k=k)
        idx = np.unique(np.concatenate([idx, np.atleast_1d(nn).astype(np.int32)]))

    return idx


def _winner_from_dist(dist_row: np.ndarray, cand_idx: np.ndarray):
    cand_nodes = reseller_node[cand_idx]
    t = dist_row[cand_nodes]
    a = reseller_attr[cand_idx]
    scores = np.array([_score_huff(aa, tt) for aa, tt in zip(a, t)], dtype=np.float64)
    if np.all(~np.isfinite(scores)):
        return NODATA_INT, np.nan
    j_local = int(np.nanargmax(scores))
    fid = int(reseller_fid[cand_idx[j_local]])
    tmin = float(t[j_local]) if np.isfinite(t[j_local]) else np.nan
    return fid, tmin


# =====================================================================
# MAIN LOOP (shortest-path per pixel source)
# =====================================================================
print("[6/8] Running Huff assignment with shortest-path travel times...")
loop_t0 = time.time()

for k, (r, c) in enumerate(zip(pix_rows, pix_cols), start=1):
    src_node = node_id[r, c]
    if src_node < 0:
        continue

    cand_walk = _candidate_idx_progressive(int(r), int(c))
    cand_walk_nodes = reseller_node[cand_walk]

    # Walk shortest-path: set time limit to slightly above farthest walk candidate by euclidean lower bound
    walk_lower_bound = np.hypot(r_rows[cand_walk] - r, r_cols[cand_walk] - c) * CELL_SIZE_METERS * max(np.nanmin(walk_friction), EPS)
    walk_limit = float(np.nanmax(walk_lower_bound) * 4.0 + 10.0) if walk_lower_bound.size > 0 else np.inf
    dist_walk = dijkstra(
        csgraph=graph_walk,
        directed=True,
        indices=[int(src_node)],
        unweighted=False,
        limit=walk_limit,
    )[0]

    fid_w, t_w = _winner_from_dist(dist_walk, cand_walk)
    best_fid_walk[r, c] = fid_w
    best_time_walk[r, c] = t_w

    cs = float(car_share[r, c])
    if cs <= CAR_SHARE_ZERO_THRESHOLD:
        best_fid_car[r, c] = fid_w
        best_time_car[r, c] = t_w
    else:
        k_extra = min(EXTRA_CANDIDATES_CAR, len(coords_rc))
        _, nn = reseller_tree.query([int(r), int(c)], k=k_extra)
        nn = np.atleast_1d(nn).astype(np.int32)
        cand_car = np.unique(np.concatenate([cand_walk, nn]))

        car_lower_bound = np.hypot(r_rows[cand_car] - r, r_cols[cand_car] - c) * CELL_SIZE_METERS * max(np.nanmin(moto_friction), EPS)
        car_limit = float(np.nanmax(car_lower_bound) * 4.0 + 10.0) if car_lower_bound.size > 0 else np.inf
        dist_car = dijkstra(
            csgraph=graph_moto,
            directed=True,
            indices=[int(src_node)],
            unweighted=False,
            limit=car_limit,
        )[0]

        fid_c, t_c = _winner_from_dist(dist_car, cand_car)
        best_fid_car[r, c] = fid_c
        best_time_car[r, c] = t_c

    if (k % PROGRESS_EVERY == 0) or (k == n_pix):
        elapsed = time.time() - loop_t0
        speed = k / max(elapsed, 1e-9)
        rem = (n_pix - k) / max(speed, 1e-9)
        total_est = elapsed + rem
        if total_est > 300:
            print(f"  {k:,}/{n_pix:,} ({100.0*k/n_pix:.1f}%) | {speed:.1f} pix/s | ETA {_eta(rem)}")
        else:
            print(f"  {k:,}/{n_pix:,} ({100.0*k/n_pix:.1f}%) | {speed:.1f} pix/s")

# =====================================================================
# OUTPUTS
# =====================================================================
print("[7/8] Writing GPKG output...")
out_rows = pix_rows.astype(np.int32)
out_cols = pix_cols.astype(np.int32)
xs = transform.c + (out_cols + 0.5) * transform.a
ys = transform.f + (out_rows + 0.5) * transform.e

df = pd.DataFrame({
    "row": out_rows,
    "col": out_cols,
    "x": xs.astype(np.float64),
    "y": ys.astype(np.float64),
    "car_share": car_share[out_rows, out_cols].astype(np.float32),
    "walk_share": walk_share[out_rows, out_cols].astype(np.float32),
    "best_fid_walk": best_fid_walk[out_rows, out_cols].astype(np.int32),
    "best_time_walk_min": best_time_walk[out_rows, out_cols].astype(np.float32),
    "best_fid_car": best_fid_car[out_rows, out_cols].astype(np.int32),
    "best_time_car_min": best_time_car[out_rows, out_cols].astype(np.float32),
})
df["best_place_id_walk"] = df["best_fid_walk"].map(fid_to_place).fillna("")
df["best_place_id_car"] = df["best_fid_car"].map(fid_to_place).fillna("")

gdf_out = gpd.GeoDataFrame(
    df,
    geometry=[Point(x, y) for x, y in zip(df["x"].to_numpy(), df["y"].to_numpy())],
    crs=crs,
)
gdf_out.to_file(OUTPUT_GPKG, layer=OUTPUT_GPKG_LAYER, driver="GPKG")
print(f"Saved: {OUTPUT_GPKG} | layer={OUTPUT_GPKG_LAYER}")

lookup = pd.DataFrame({
    "fid": reseller_fid,
    "place_id": reseller_place,
    "attractiveness": reseller_attr,
    "row": r_rows,
    "col": r_cols,
}).drop_duplicates(subset=["fid"])
lookup.to_csv(OUTPUT_LOOKUP_CSV, index=False)
print(f"Saved lookup: {OUTPUT_LOOKUP_CSV}")

print("[8/8] Writing support rasters...")
if SAVE_SUPPORT_RASTERS:
    _write_float_raster(OUTPUT_RASTER_CAR_SHARE, car_share, ref_profile)
    _write_float_raster(OUTPUT_RASTER_WALK_SHARE, walk_share, ref_profile)
    _write_int_raster(OUTPUT_RASTER_BEST_FID_WALK, best_fid_walk.astype(np.float32), ref_profile)
    _write_float_raster(OUTPUT_RASTER_BEST_TIME_WALK, best_time_walk, ref_profile)
    _write_int_raster(OUTPUT_RASTER_BEST_FID_CAR, best_fid_car.astype(np.float32), ref_profile)
    _write_float_raster(OUTPUT_RASTER_BEST_TIME_CAR, best_time_car, ref_profile)
    print("Support rasters saved.")
else:
    print("Support rasters skipped.")

valid_walk = np.isfinite(best_time_walk[inhabited])
valid_car = np.isfinite(best_time_car[inhabited])
print("\n=== SUMMARY ===")
print(f"Pixels processed: {n_pix:,}")
print(f"Walk winners found: {int(valid_walk.sum()):,} ({100.0 * valid_walk.mean():.2f}%)")
print(f"Car winners found : {int(valid_car.sum()):,} ({100.0 * valid_car.mean():.2f}%)")
if valid_walk.any():
    w = best_time_walk[inhabited]
    print(f"Walk time min/median/max (min): {np.nanmin(w):.2f} / {np.nanmedian(w):.2f} / {np.nanmax(w):.2f}")
if valid_car.any():
    c = best_time_car[inhabited]
    print(f"Car  time min/median/max (min): {np.nanmin(c):.2f} / {np.nanmedian(c):.2f} / {np.nanmax(c):.2f}")

print(f"\nDone in {_eta(time.time() - t0)}")